# 🤖 AI Robotic Arm — Exploratory Analysis & Model Evaluation

ใช้ notebook นี้สำหรับ:
- ทดสอบ YOLO บน test images
- Visualize IK workspace
- วิเคราะห์ detection performance
- Debug coordinate mapping

In [ ]:
import sys
sys.path.insert(0, '..')   # add project root to path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['figure.facecolor'] = '#16213e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
print('✓ Imports ready')

## 1. Visualize Arm Workspace (IK Reachability Map)

In [ ]:
from src.robotics.kinematics import IKSolver

solver = IKSolver()

# Sample workspace on XY plane (Z=0.05)
xs = np.linspace(-0.30, 0.30, 60)
ys = np.linspace(0.05, 0.40, 60)
XX, YY = np.meshgrid(xs, ys)
reachable = np.zeros_like(XX, dtype=bool)

for i in range(len(ys)):
    for j in range(len(xs)):
        xyz = np.array([XX[i,j], YY[i,j], 0.05])
        reachable[i,j] = solver.is_reachable(xyz)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(XX, YY, reachable.astype(float),
            levels=1, colors=['#2d2d2d', '#00b4d8'], alpha=0.8)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('🤖 Arm Reachability Map (Z=0.05m)', color='white', fontsize=14)
ax.set_aspect('equal')
ax.grid(color='#333', linestyle='--', linewidth=0.5)
blue_patch = mpatches.Patch(color='#00b4d8', label='Reachable')
gray_patch = mpatches.Patch(color='#2d2d2d', label='Unreachable')
ax.legend(handles=[blue_patch, gray_patch])
plt.tight_layout()
plt.show()
print(f'Reachable area: {reachable.sum()/reachable.size*100:.1f}%')

## 2. IK Round-Trip Accuracy Test

In [ ]:
# Test IK accuracy across 200 random reachable targets
errors_mm = []
n_tests = 200
n_failed = 0

np.random.seed(42)
for _ in range(n_tests):
    xyz = np.array([
        np.random.uniform(-0.20, 0.20),
        np.random.uniform(0.10, 0.35),
        np.random.uniform(0.00, 0.20),
    ])
    if not solver.is_reachable(xyz):
        continue
    
    angles = solver.solve(xyz)
    if angles is None:
        n_failed += 1
        continue
    
    ee = solver.forward(angles)
    errors_mm.append(np.linalg.norm(ee - xyz) * 1000)

errors_mm = np.array(errors_mm)
print(f'Tests: {len(errors_mm)} solved / {n_failed} failed')
print(f'Mean error: {errors_mm.mean():.3f} mm')
print(f'Max  error: {errors_mm.max():.3f} mm')
print(f'< 1mm:      {(errors_mm < 1.0).mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(errors_mm, bins=40, color='#00b4d8', edgecolor='#0077a8', alpha=0.85)
ax.axvline(errors_mm.mean(), color='#ff6b6b', linewidth=2, label=f'Mean={errors_mm.mean():.2f}mm')
ax.set_xlabel('Round-trip error (mm)')
ax.set_ylabel('Count')
ax.set_title('IK Accuracy Distribution', color='white', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 3. YOLO Inference on Test Image

In [ ]:
import cv2
from src.vision.detect import ObjectDetector
from IPython.display import display
from PIL import Image

detector = ObjectDetector()

# Load a test image (replace with your own)
# img_path = '../data/images/test/sample.jpg'
# frame = cv2.imread(img_path)

# --- OR use webcam frame ---
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    result = detector.detect(frame)
    vis    = detector.annotate_frame(frame, result)
    
    print(f'Detected {result.count} objects in {result.inference_ms:.1f}ms')
    for d in result.detections:
        print(f'  {d.class_name:15s} conf={d.confidence:.2f}  center={d.center_px}')
    
    rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(rgb))
else:
    print('No camera available — load a test image instead')

## 4. Decision Engine Simulation

In [ ]:
import time
from src.vision.detect import Detection, DetectionResult
from src.logic.decision import DecisionEngine, SORT_MAP, DROP_ZONES

# Show the bin routing table
print('🗑 Bin Routing Table:')
print('-' * 35)
for obj, bin_ in sorted(SORT_MAP.items()):
    zone = DROP_ZONES[bin_]
    print(f'  {obj:15s} → {bin_:10s}  drop={zone}')

print('\n📍 Drop Zone Positions:')
print('-' * 35)
for bin_, xyz in DROP_ZONES.items():
    print(f'  {bin_:12s}  X={xyz[0]:+.2f}m  Y={xyz[1]:.2f}m  Z={xyz[2]:.2f}m')